In [ ]:
# ==========================================
# DAY 2 — SYSTEM CHECK
# ==========================================

import torch
import os

print("====================================")
print("DAY 2 — SYSTEM CHECK")
print("====================================")

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
else:
    print("WARNING: GPU is not available.")

PROJECT_ROOT = "/content/drive/MyDrive/SLM_KD_Research"

print("\nProject root:")
print(PROJECT_ROOT)

print("\nBaseline checkpoint exists:",
      os.path.exists(
          os.path.join(
              PROJECT_ROOT,
              "models",
              "baseline_student",
              "final",
              "model.safetensors"
          )
      ))

DAY 2 — SYSTEM CHECK
PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
CUDA version: 12.8

Project root:
/content/drive/MyDrive/SLM_KD_Research

Baseline checkpoint exists: False


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

PROJECT_ROOT = "/content/drive/MyDrive/SLM_KD_Research"

BASELINE_CHECKPOINT = os.path.join(
    PROJECT_ROOT,
    "models",
    "baseline_student",
    "final"
)

print("Project folder exists:",
      os.path.exists(PROJECT_ROOT))

print("\nBaseline checkpoint exists:",
      os.path.exists(BASELINE_CHECKPOINT))

print("\nCheckpoint files:")

if os.path.exists(BASELINE_CHECKPOINT):
    for item in os.listdir(BASELINE_CHECKPOINT):
        print("-", item)

Project folder exists: True

Baseline checkpoint exists: True

Checkpoint files:
- config.json
- model.safetensors
- tokenizer_config.json
- tokenizer.json


In [ ]:
!pip install -q transformers datasets scikit-learn

In [ ]:
import os
import json
import time
import random

import numpy as np
import torch
import torch.nn.functional as F

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding
)
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, f1_score

print("All imports successful.")

All imports successful.


In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Random seed:", SEED)

Random seed: 42


In [ ]:
# ==========================================
# DAY 2 — EXPERIMENT CONFIGURATION
# ==========================================

PROJECT_ROOT = "/content/drive/MyDrive/SLM_KD_Research"

# Student model
STUDENT_MODEL = "google/bert_uncased_L-2_H-128_A-2"

# Dataset
DATASET_NAME = "nyu-mll/glue"
DATASET_CONFIG = "sst2"

# Data settings
MAX_LENGTH = 128
BATCH_SIZE = 16

# Training settings
LEARNING_RATE = 2e-5
EPOCHS = 3

# Knowledge Distillation settings
TEMPERATURE = 1.5
ALPHA = 0.4

# Device
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("====================================")
print("DAY 2 — CONFIGURATION")
print("====================================")

print("Student model:", STUDENT_MODEL)
print("Dataset:", DATASET_NAME, DATASET_CONFIG)
print("Max length:", MAX_LENGTH)
print("Batch size:", BATCH_SIZE)
print("Learning rate:", LEARNING_RATE)
print("Epochs:", EPOCHS)
print("KD temperature:", TEMPERATURE)
print("KD alpha:", ALPHA)
print("Device:", DEVICE)

DAY 2 — CONFIGURATION
Student model: google/bert_uncased_L-2_H-128_A-2
Dataset: nyu-mll/glue sst2
Max length: 128
Batch size: 16
Learning rate: 2e-05
Epochs: 3
KD temperature: 1.5
KD alpha: 0.4
Device: cuda


In [ ]:
# ==========================================
# STEP 8 — LOAD SST-2 DATASET
# ==========================================

dataset = load_dataset(
    DATASET_NAME,
    DATASET_CONFIG
)

print(dataset)

README.md:   0%|          | 0.00/35.3k [00:00<?, ?B/s]

sst2/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.11MB            

sst2/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

sst2/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 72.8kB            

sst2/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

sst2/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 872
    })
    test: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 1821
    })
})


In [ ]:
# ==========================================
# STEP 9 — VERIFY DATASET LABELS
# ==========================================

print("Train examples:", len(dataset["train"]))
print("Validation examples:", len(dataset["validation"]))
print("Test examples:", len(dataset["test"]))

print("\nValidation labels:")
print(set(dataset["validation"]["label"]))

print("\nTest labels:")
print(set(dataset["test"]["label"]))

Train examples: 67349
Validation examples: 872
Test examples: 1821

Validation labels:
{0, 1}

Test labels:
{-1}


In [ ]:
# ==========================================
# STEP 10 — LOAD STUDENT TOKENIZER
# ==========================================

student_tokenizer = AutoTokenizer.from_pretrained(
    STUDENT_MODEL
)

print("Student tokenizer loaded successfully.")
print("Student model:", STUDENT_MODEL)
print("Vocabulary size:", student_tokenizer.vocab_size)

config.json:   0%|          | 0.00/382 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

Student tokenizer loaded successfully.
Student model: google/bert_uncased_L-2_H-128_A-2
Vocabulary size: 30522


In [ ]:
# ==========================================
# STEP 11 — TEACHER MODEL
# ==========================================

TEACHER_MODEL = "textattack/bert-base-uncased-SST-2"

print("Teacher model:", TEACHER_MODEL)


Teacher model: textattack/bert-base-uncased-SST-2


In [ ]:
# ==========================================
# STEP 12 — LOAD TEACHER TOKENIZER
# ==========================================

teacher_tokenizer = AutoTokenizer.from_pretrained(
    TEACHER_MODEL
)

print("Teacher tokenizer loaded successfully.")
print("Vocabulary size:", teacher_tokenizer.vocab_size)

config.json:   0%|          | 0.00/477 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Teacher tokenizer loaded successfully.
Vocabulary size: 30522


In [ ]:
# ==========================================
# STEP 13 — LOAD TEACHER MODEL
# ==========================================

teacher_model = AutoModelForSequenceClassification.from_pretrained(
    TEACHER_MODEL,
    num_labels=2
)

teacher_model = teacher_model.to(DEVICE)

print("Teacher model loaded successfully.")
print("Teacher device:", next(teacher_model.parameters()).device)

teacher_parameters = sum(
    p.numel() for p in teacher_model.parameters()
)

print("Teacher parameters:", teacher_parameters)

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  438MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Teacher model loaded successfully.
Teacher device: cuda:0
Teacher parameters: 109483778


In [ ]:
# ==========================================
# STEP 14A — TEACHER VALIDATION DATALOADER
# ==========================================

validation_dataset = dataset["validation"]

def tokenize_teacher(example):
    tokenized_example = teacher_tokenizer(
        example["sentence"],
        truncation=True,
        max_length=MAX_LENGTH
    )
    tokenized_example["label"] = example["label"] # Include the label in the tokenized example
    return tokenized_example

teacher_validation = validation_dataset.map(
    tokenize_teacher,
    batched=False
)

teacher_validation = teacher_validation.remove_columns(
    ["sentence", "idx"]
)

teacher_validation.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "label"]
)

teacher_collator = DataCollatorWithPadding(
    tokenizer=teacher_tokenizer
)

teacher_validation_loader = DataLoader(
    teacher_validation,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=teacher_collator
)

print("Teacher validation examples:",
      len(teacher_validation))

print("Teacher validation batches:",
      len(teacher_validation_loader))

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

Teacher validation examples: 872
Teacher validation batches: 55


In [ ]:
# ==========================================
# STEP 14B — FIX TEACHER VALIDATION DATA
# ==========================================

validation_dataset = dataset["validation"]

def tokenize_teacher(example):
    return teacher_tokenizer(
        example["sentence"],
        truncation=True,
        max_length=MAX_LENGTH
    )

teacher_validation = validation_dataset.map(
    tokenize_teacher,
    batched=True
)

# Rename label → labels for Hugging Face model compatibility
teacher_validation = teacher_validation.rename_column(
    "label",
    "labels"
)

teacher_validation = teacher_validation.remove_columns(
    ["sentence", "idx"]
)

teacher_validation.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

teacher_validation_loader = DataLoader(
    teacher_validation,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=DataCollatorWithPadding(
        tokenizer=teacher_tokenizer
    )
)

print("Teacher validation examples:",
      len(teacher_validation))

print("Teacher validation batches:",
      len(teacher_validation_loader))

# Check one batch
sample_batch = next(iter(teacher_validation_loader))

print("\nBatch keys:")
print(sample_batch.keys())

print("\nLabel shape:")
print(sample_batch["labels"].shape)

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

Teacher validation examples: 872
Teacher validation batches: 55

Batch keys:
KeysView({'labels': tensor([1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 0, 1]), 'input_ids': tensor([[  101,  2009,  1005,  1055,  1037, 11951,  1998,  2411, 12473,  4990,
          1012,   102,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0],
        [  101,  4895, 10258,  2378,  8450,  2135, 21657,  1998,  7143,   102,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0],
        [  101,  4473,  2149,  2000,  3246,  2008, 13401,  2003, 22303,  2000,
         28866,  1037,  2350,  2476,  2004,  1037,  3293,  2664,  1999, 15338,
          3512, 12127,  1012,   102,    

In [ ]:
# ==========================================
# STEP 15 — TEACHER VALIDATION EVALUATION
# ==========================================

teacher_model.eval()

teacher_predictions = []
teacher_labels = []
teacher_validation_losses = []

with torch.no_grad():

    for batch in teacher_validation_loader:

        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        outputs = teacher_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        logits = outputs.logits
        loss = outputs.loss

        predictions = torch.argmax(logits, dim=-1)

        teacher_validation_losses.append(loss.item())

        teacher_predictions.extend(
            predictions.cpu().numpy()
        )

        teacher_labels.extend(
            labels.cpu().numpy()
        )

# Calculate metrics
teacher_val_loss = sum(teacher_validation_losses) / len(
    teacher_validation_losses
)

teacher_val_accuracy = accuracy_score(
    teacher_labels,
    teacher_predictions
)

teacher_val_f1 = f1_score(
    teacher_labels,
    teacher_predictions
)

print("====================================")
print("TEACHER VALIDATION RESULTS")
print("====================================")

print("Teacher model:")
print(TEACHER_MODEL)

print("Validation samples:")
print(len(teacher_labels))

print("Validation loss:")
print(teacher_val_loss)

print("Validation accuracy:")
print(teacher_val_accuracy)

print("Validation F1:")
print(teacher_val_f1)

TEACHER VALIDATION RESULTS
Teacher model:
textattack/bert-base-uncased-SST-2
Validation samples:
872
Validation loss:
0.2797187480652197
Validation accuracy:
0.9243119266055045
Validation F1:
0.9263392857142857


In [ ]:
# ==========================================
# STEP 16 — FREEZE TEACHER
# ==========================================

teacher_model.eval()

for param in teacher_model.parameters():
    param.requires_grad = False

# Verify
trainable_teacher_params = sum(
    p.numel()
    for p in teacher_model.parameters()
    if p.requires_grad
)

total_teacher_params = sum(
    p.numel()
    for p in teacher_model.parameters()
)

print("====================================")
print("TEACHER FREEZE CHECK")
print("====================================")

print("Total teacher parameters:",
      total_teacher_params)

print("Trainable teacher parameters:",
      trainable_teacher_params)

print("Teacher training mode:",
      teacher_model.training)

print("Teacher frozen:",
      trainable_teacher_params == 0)

TEACHER FREEZE CHECK
Total teacher parameters: 109483778
Trainable teacher parameters: 0
Teacher training mode: False
Teacher frozen: True


In [ ]:
# ==========================================
# STEP 17 — CREATE FRESH KD STUDENT
# ==========================================

from transformers import AutoModelForSequenceClassification

print("Loading fresh KD student...")

kd_student_model = AutoModelForSequenceClassification.from_pretrained(
    STUDENT_MODEL,
    num_labels=2
)

kd_student_model = kd_student_model.to(DEVICE)

# Count parameters
total_student_params = sum(
    p.numel()
    for p in kd_student_model.parameters()
)

trainable_student_params = sum(
    p.numel()
    for p in kd_student_model.parameters()
    if p.requires_grad
)

print("====================================")
print("KD STUDENT CHECK")
print("====================================")

print("Student model:")
print(STUDENT_MODEL)

print("Total parameters:")
print(total_student_params)

print("Trainable parameters:")
print(trainable_student_params)

print("Device:")
print(next(kd_student_model.parameters()).device)

print("Training mode:")
print(kd_student_model.training)

Loading fresh KD student...


model.safetensors: reconstructing file:   0%|          |  0.00B / 17.7MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/39 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google/bert_uncased_L-2_H-128_A-2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


KD STUDENT CHECK
Student model:
google/bert_uncased_L-2_H-128_A-2
Total parameters:
4386178
Trainable parameters:
4386178
Device:
cuda:0
Training mode:
False


In [ ]:
# ==========================================
# STEP 18 — CREATE KD TRAINING DATASET
# ==========================================

from torch.utils.data import Dataset, DataLoader

class KDDataset(Dataset):

    def __init__(
        self,
        hf_dataset,
        student_tokenizer,
        teacher_tokenizer,
        max_length
    ):

        self.sentences = hf_dataset["sentence"]
        self.labels = hf_dataset["label"]

        self.student_tokenizer = student_tokenizer
        self.teacher_tokenizer = teacher_tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, index):

        sentence = self.sentences[index]
        label = self.labels[index]

        student_encoding = self.student_tokenizer(
            sentence,
            truncation=True,
            max_length=self.max_length
        )

        teacher_encoding = self.teacher_tokenizer(
            sentence,
            truncation=True,
            max_length=self.max_length
        )

        return {
            "student_input_ids": student_encoding["input_ids"],
            "student_attention_mask": student_encoding["attention_mask"],
            "teacher_input_ids": teacher_encoding["input_ids"],
            "teacher_attention_mask": teacher_encoding["attention_mask"],
            "labels": label
        }


kd_train_dataset = KDDataset(
    dataset["train"],
    student_tokenizer,
    teacher_tokenizer,
    MAX_LENGTH
)

print("====================================")
print("KD DATASET CHECK")
print("====================================")

print("KD training examples:")
print(len(kd_train_dataset))

sample = kd_train_dataset[0]

print("\nSample keys:")
print(sample.keys())

print("\nSample label:")
print(sample["labels"])

print("\nStudent input length:")
print(len(sample["student_input_ids"]))

print("\nTeacher input length:")
print(len(sample["teacher_input_ids"]))

KD DATASET CHECK
KD training examples:
67349

Sample keys:
dict_keys(['student_input_ids', 'student_attention_mask', 'teacher_input_ids', 'teacher_attention_mask', 'labels'])

Sample label:
0

Student input length:
10

Teacher input length:
10


In [ ]:
# ==========================================
# STEP 18B — FIX KD DATASET LABEL
# ==========================================

class KDDataset(Dataset):

    def __init__(
        self,
        hf_dataset,
        student_tokenizer,
        teacher_tokenizer,
        max_length
    ):

        self.sentences = hf_dataset["sentence"]
        self.labels = hf_dataset["label"]

        self.student_tokenizer = student_tokenizer
        self.teacher_tokenizer = teacher_tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, index):

        sentence = self.sentences[index]
        label = self.labels[index]

        student_encoding = self.student_tokenizer(
            sentence,
            truncation=True,
            max_length=self.max_length
        )

        teacher_encoding = self.teacher_tokenizer(
            sentence,
            truncation=True,
            max_length=self.max_length
        )

        return {
            "student_input_ids": student_encoding["input_ids"],
            "student_attention_mask": student_encoding["attention_mask"],
            "teacher_input_ids": teacher_encoding["input_ids"],
            "teacher_attention_mask": teacher_encoding["attention_mask"],
            "labels": label
        }


kd_train_dataset = KDDataset(
    dataset["train"],
    student_tokenizer,
    teacher_tokenizer,
    MAX_LENGTH
)

sample = kd_train_dataset[0]

print("====================================")
print("KD DATASET FIX CHECK")
print("====================================")

print("Number of examples:")
print(len(kd_train_dataset))

print("\nSample keys:")
print(sample.keys())

print("\nSample label:")
print(sample["labels"])

KD DATASET FIX CHECK
Number of examples:
67349

Sample keys:
dict_keys(['student_input_ids', 'student_attention_mask', 'teacher_input_ids', 'teacher_attention_mask', 'labels'])

Sample label:
0


In [ ]:
print("====================================")
print("KD DATASET DIAGNOSTIC")
print("====================================")

sample = kd_train_dataset[0]

print("Sample type:")
print(type(sample))

print("\nSample:")
print(sample)

print("\nSample keys:")
print(list(sample.keys()))

print("\nDoes labels exist?")
print("labels" in sample)

print("\nLabel value:")
print(sample["labels"])

KD DATASET DIAGNOSTIC
Sample type:
<class 'dict'>

Sample:
{'student_input_ids': [101, 5342, 2047, 3595, 8496, 2013, 1996, 18643, 3197, 102], 'student_attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'teacher_input_ids': [101, 5342, 2047, 3595, 8496, 2013, 1996, 18643, 3197, 102], 'teacher_attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'labels': 0}

Sample keys:
['student_input_ids', 'student_attention_mask', 'teacher_input_ids', 'teacher_attention_mask', 'labels']

Does labels exist?
True

Label value:
0


In [ ]:
# ==========================================
# STEP 19 — KD COLLATOR + DATALOADER
# ==========================================

class KDCollator:

    def __init__(
        self,
        student_tokenizer,
        teacher_tokenizer
    ):
        self.student_tokenizer = student_tokenizer
        self.teacher_tokenizer = teacher_tokenizer

    def __call__(self, batch):

        student_features = [
            {
                "input_ids": item["student_input_ids"],
                "attention_mask": item["student_attention_mask"]
            }
            for item in batch
        ]

        teacher_features = [
            {
                "input_ids": item["teacher_input_ids"],
                "attention_mask": item["teacher_attention_mask"]
            }
            for item in batch
        ]

        student_batch = self.student_tokenizer.pad(
            student_features,
            return_tensors="pt"
        )

        teacher_batch = self.teacher_tokenizer.pad(
            teacher_features,
            return_tensors="pt"
        )

        labels = torch.tensor(
            [item["labels"] for item in batch],
            dtype=torch.long
        )

        return {
            "student_input_ids": student_batch["input_ids"],
            "student_attention_mask": student_batch["attention_mask"],
            "teacher_input_ids": teacher_batch["input_ids"],
            "teacher_attention_mask": teacher_batch["attention_mask"],
            "labels": labels
        }


kd_collator = KDCollator(
    student_tokenizer,
    teacher_tokenizer
)

kd_train_loader = DataLoader(
    kd_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=kd_collator
)

print("====================================")
print("KD DATALOADER CHECK")
print("====================================")

print("Training examples:")
print(len(kd_train_dataset))

print("Batch size:")
print(BATCH_SIZE)

print("Number of batches:")
print(len(kd_train_loader))

# Check one real batch
kd_batch = next(iter(kd_train_loader))

print("\nBatch keys:")
print(kd_batch.keys())

print("\nStudent input shape:")
print(kd_batch["student_input_ids"].shape)

print("\nTeacher input shape:")
print(kd_batch["teacher_input_ids"].shape)

print("\nLabels shape:")
print(kd_batch["labels"].shape)

print("\nLabels:")
print(kd_batch["labels"])

KD DATALOADER CHECK
Training examples:
67349
Batch size:
16
Number of batches:
4210

Batch keys:
dict_keys(['student_input_ids', 'student_attention_mask', 'teacher_input_ids', 'teacher_attention_mask', 'labels'])

Student input shape:
torch.Size([16, 51])

Teacher input shape:
torch.Size([16, 51])

Labels shape:
torch.Size([16])

Labels:
tensor([1, 0, 1, 0, 1, 1, 1, 0, 1, 0, 0, 0, 1, 0, 1, 0])


In [ ]:
# ==========================================
# STEP 20 — KD FORWARD-PASS SANITY CHECK
# ==========================================

import torch.nn.functional as F

# Put models in correct modes
teacher_model.eval()
kd_student_model.train()

# Get one batch
kd_batch = next(iter(kd_train_loader))

# Move inputs to GPU
student_input_ids = kd_batch["student_input_ids"].to(DEVICE)
student_attention_mask = kd_batch["student_attention_mask"].to(DEVICE)

teacher_input_ids = kd_batch["teacher_input_ids"].to(DEVICE)
teacher_attention_mask = kd_batch["teacher_attention_mask"].to(DEVICE)

labels = kd_batch["labels"].to(DEVICE)

# ------------------------------------------
# Teacher forward pass
# ------------------------------------------

with torch.no_grad():

    teacher_outputs = teacher_model(
        input_ids=teacher_input_ids,
        attention_mask=teacher_attention_mask
    )

    teacher_logits = teacher_outputs.logits

# ------------------------------------------
# Student forward pass
# ------------------------------------------

student_outputs = kd_student_model(
    input_ids=student_input_ids,
    attention_mask=student_attention_mask
)

student_logits = student_outputs.logits

# ------------------------------------------
# Check shapes
# ------------------------------------------

print("====================================")
print("KD FORWARD PASS CHECK")
print("====================================")

print("Teacher logits shape:")
print(teacher_logits.shape)

print("Student logits shape:")
print(student_logits.shape)

print("Labels shape:")
print(labels.shape)

# ------------------------------------------
# Calculate KD loss
# ------------------------------------------

temperature = TEMPERATURE

student_log_probs = F.log_softmax(
    student_logits / temperature,
    dim=-1
)

teacher_probs = F.softmax(
    teacher_logits / temperature,
    dim=-1
)

kd_loss = F.kl_div(
    student_log_probs,
    teacher_probs,
    reduction="batchmean"
)

kd_loss = kd_loss * (temperature ** 2)

# Normal supervised CE loss
ce_loss = F.cross_entropy(
    student_logits,
    labels
)

# Combined loss
total_loss = (
    ALPHA * kd_loss
    + (1 - ALPHA) * ce_loss
)

print("\nTemperature:")
print(TEMPERATURE)

print("Alpha:")
print(ALPHA)

print("\nKD loss:")
print(kd_loss.item())

print("CE loss:")
print(ce_loss.item())

print("Combined loss:")
print(total_loss.item())

print("\nDevice check:")
print("Teacher logits:", teacher_logits.device)
print("Student logits:", student_logits.device)
print("Labels:", labels.device)

KD FORWARD PASS CHECK
Teacher logits shape:
torch.Size([16, 2])
Student logits shape:
torch.Size([16, 2])
Labels shape:
torch.Size([16])

Temperature:
1.0
Alpha:
0.3

KD loss:
0.6769349575042725
CE loss:
0.7045192122459412
Combined loss:
0.696243941783905

Device check:
Teacher logits: cuda:0
Student logits: cuda:0
Labels: cuda:0


In [ ]:
# ==========================================
# STEP 21 — KNOWLEDGE DISTILLATION TRAINING
# ==========================================

import time
import torch.nn.functional as F
from torch.optim import AdamW

# ------------------------------------------
# Training configuration
# ------------------------------------------

temperature = TEMPERATURE
alpha = ALPHA

# Fresh optimizer for the fresh KD student
optimizer = AdamW(
    kd_student_model.parameters(),
    lr=LEARNING_RATE
)

kd_training_history = []

print("====================================")
print("STARTING KD TRAINING")
print("====================================")

print("Student:", STUDENT_MODEL)
print("Teacher:", TEACHER_MODEL)
print("Epochs:", EPOCHS)
print("Batch size:", BATCH_SIZE)
print("Learning rate:", LEARNING_RATE)
print("Temperature:", temperature)
print("Alpha:", alpha)
print("Training examples:", len(kd_train_dataset))
print("Training batches:", len(kd_train_loader))
print("Device:", DEVICE)

for epoch in range(EPOCHS):

    epoch_start_time = time.time()

    kd_student_model.train()
    teacher_model.eval()

    total_epoch_loss = 0.0
    total_epoch_ce_loss = 0.0
    total_epoch_kd_loss = 0.0

    for step, batch in enumerate(kd_train_loader):

        # ----------------------------------
        # Move data to GPU
        # ----------------------------------

        student_input_ids = batch[
            "student_input_ids"
        ].to(DEVICE)

        student_attention_mask = batch[
            "student_attention_mask"
        ].to(DEVICE)

        teacher_input_ids = batch[
            "teacher_input_ids"
        ].to(DEVICE)

        teacher_attention_mask = batch[
            "teacher_attention_mask"
        ].to(DEVICE)

        labels = batch[
            "labels"
        ].to(DEVICE)

        # ----------------------------------
        # Teacher forward pass
        # ----------------------------------

        with torch.no_grad():

            teacher_outputs = teacher_model(
                input_ids=teacher_input_ids,
                attention_mask=teacher_attention_mask
            )

            teacher_logits = teacher_outputs.logits

        # ----------------------------------
        # Student forward pass
        # ----------------------------------

        student_outputs = kd_student_model(
            input_ids=student_input_ids,
            attention_mask=student_attention_mask
        )

        student_logits = student_outputs.logits

        # ----------------------------------
        # Knowledge Distillation Loss
        # ----------------------------------

        student_log_probs = F.log_softmax(
            student_logits / temperature,
            dim=-1
        )

        teacher_probs = F.softmax(
            teacher_logits / temperature,
            dim=-1
        )

        kd_loss = F.kl_div(
            student_log_probs,
            teacher_probs,
            reduction="batchmean"
        )

        kd_loss = kd_loss * (temperature ** 2)

        # ----------------------------------
        # Normal supervised loss
        # ----------------------------------

        ce_loss = F.cross_entropy(
            student_logits,
            labels
        )

        # ----------------------------------
        # Combined KD loss
        # ----------------------------------

        total_loss = (
            alpha * kd_loss
            + (1 - alpha) * ce_loss
        )

        # ----------------------------------
        # Backpropagation
        # ----------------------------------

        optimizer.zero_grad()

        total_loss.backward()

        optimizer.step()

        # ----------------------------------
        # Track losses
        # ----------------------------------

        total_epoch_loss += total_loss.item()
        total_epoch_ce_loss += ce_loss.item()
        total_epoch_kd_loss += kd_loss.item()

        # ----------------------------------
        # Progress display
        # ----------------------------------

        if (step + 1) % 500 == 0:

            print(
                f"Epoch {epoch + 1}/{EPOCHS} | "
                f"Step {step + 1}/{len(kd_train_loader)} | "
                f"Loss {total_loss.item():.4f}"
            )

    # --------------------------------------
    # Epoch averages
    # --------------------------------------

    avg_total_loss = (
        total_epoch_loss / len(kd_train_loader)
    )

    avg_ce_loss = (
        total_epoch_ce_loss / len(kd_train_loader)
    )

    avg_kd_loss = (
        total_epoch_kd_loss / len(kd_train_loader)
    )

    epoch_time = time.time() - epoch_start_time

    epoch_result = {
        "epoch": epoch + 1,
        "total_loss": avg_total_loss,
        "ce_loss": avg_ce_loss,
        "kd_loss": avg_kd_loss,
        "time_seconds": epoch_time
    }

    kd_training_history.append(epoch_result)

    print("\n------------------------------------")
    print(f"Epoch {epoch + 1} completed")
    print("------------------------------------")

    print("Average total loss:", avg_total_loss)
    print("Average CE loss:", avg_ce_loss)
    print("Average KD loss:", avg_kd_loss)
    print("Time:", epoch_time, "seconds")

print("\n====================================")
print("KD TRAINING COMPLETE")
print("====================================")

for result in kd_training_history:
    print(result)

STARTING KD TRAINING
Student: google/bert_uncased_L-2_H-128_A-2
Teacher: textattack/bert-base-uncased-SST-2
Epochs: 3
Batch size: 16
Learning rate: 2e-05
Temperature: 1.5
Alpha: 0.4
Training examples: 67349
Training batches: 4210
Device: cuda
Epoch 1/3 | Step 500/4210 | Loss 0.1016
Epoch 1/3 | Step 1000/4210 | Loss 0.3413
Epoch 1/3 | Step 1500/4210 | Loss 0.1614
Epoch 1/3 | Step 2000/4210 | Loss 0.1538
Epoch 1/3 | Step 2500/4210 | Loss 0.4330
Epoch 1/3 | Step 3000/4210 | Loss 0.1478
Epoch 1/3 | Step 3500/4210 | Loss 0.2114
Epoch 1/3 | Step 4000/4210 | Loss 0.2934

------------------------------------
Epoch 1 completed
------------------------------------
Average total loss: 0.2681914729608174
Average CE loss: 0.2238379497934938
Average KD loss: 0.33472173756913165
Time: 197.87132811546326 seconds
Epoch 2/3 | Step 500/4210 | Loss 0.5859
Epoch 2/3 | Step 1000/4210 | Loss 0.2045
Epoch 2/3 | Step 1500/4210 | Loss 0.3210
Epoch 2/3 | Step 2000/4210 | Loss 0.5137
Epoch 2/3 | Step 2500/4210 | 

In [ ]:
# ==========================================
# STEP 22 — KD STUDENT VALIDATION
# ==========================================

from sklearn.metrics import accuracy_score, f1_score

kd_student_model.eval()

kd_validation_dataset = dataset["validation"]

def tokenize_kd_validation(example):
    return student_tokenizer(
        example["sentence"],
        truncation=True,
        max_length=MAX_LENGTH
    )

kd_validation_dataset = kd_validation_dataset.map(
    tokenize_kd_validation,
    batched=True
)

kd_validation_dataset = kd_validation_dataset.remove_columns(
    ["sentence", "idx"]
)

kd_validation_dataset = kd_validation_dataset.rename_column(
    "label",
    "labels"
)

kd_validation_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

kd_validation_loader = DataLoader(
    kd_validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=DataCollatorWithPadding(
        tokenizer=student_tokenizer
    )
)

kd_predictions = []
kd_labels = []
kd_validation_losses = []

with torch.no_grad():

    for batch in kd_validation_loader:

        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        outputs = kd_student_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        predictions = torch.argmax(
            outputs.logits,
            dim=-1
        )

        kd_validation_losses.append(
            outputs.loss.item()
        )

        kd_predictions.extend(
            predictions.cpu().numpy()
        )

        kd_labels.extend(
            labels.cpu().numpy()
        )

kd_val_loss = sum(kd_validation_losses) / len(
    kd_validation_losses
)

kd_val_accuracy = accuracy_score(
    kd_labels,
    kd_predictions
)

kd_val_f1 = f1_score(
    kd_labels,
    kd_predictions
)

print("====================================")
print("KD STUDENT VALIDATION RESULTS")
print("====================================")

print("Student model:")
print(STUDENT_MODEL)

print("Validation samples:")
print(len(kd_labels))

print("Validation loss:")
print(kd_val_loss)

print("Validation accuracy:")
print(kd_val_accuracy)

print("Validation F1:")
print(kd_val_f1)

KD STUDENT VALIDATION RESULTS
Student model:
google/bert_uncased_L-2_H-128_A-2
Validation samples:
872
Validation loss:
0.6292748449878259
Validation accuracy:
0.819954128440367
Validation F1:
0.8302702702702702


In [ ]:
# ==========================================
# STEP 23 — SAVE KD STUDENT CHECKPOINT
# ==========================================

import os

KD_CHECKPOINT_DIR = os.path.join(
    PROJECT_ROOT,
    "models",
    "kd",
    "student_epoch3"
)

os.makedirs(
    KD_CHECKPOINT_DIR,
    exist_ok=True
)

kd_student_model.save_pretrained(
    KD_CHECKPOINT_DIR
)

student_tokenizer.save_pretrained(
    KD_CHECKPOINT_DIR
)

print("====================================")
print("KD CHECKPOINT SAVED")
print("====================================")

print("Checkpoint path:")
print(KD_CHECKPOINT_DIR)

print("\nFiles saved:")

for filename in sorted(
    os.listdir(KD_CHECKPOINT_DIR)
):
    print(filename)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

KD CHECKPOINT SAVED
Checkpoint path:
/content/drive/MyDrive/SLM_KD_Research/models/kd/student_epoch3

Files saved:
config.json
model.safetensors
tokenizer.json
tokenizer_config.json


In [ ]:
# ==========================================
# STEP 24 — SAVE KD RESULTS
# ==========================================

import json
import os

RESULTS_DIR = os.path.join(
    PROJECT_ROOT,
    "results"
)

os.makedirs(
    RESULTS_DIR,
    exist_ok=True
)

# ------------------------------------------
# Save final KD validation result
# ------------------------------------------

kd_results = {
    "experiment": "E2_KD_student",
    "teacher_model": TEACHER_MODEL,
    "student_model": STUDENT_MODEL,
    "dataset": "SST-2",
    "split": "validation",
    "validation_samples": len(kd_labels),
    "max_length": MAX_LENGTH,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "epochs": EPOCHS,
    "temperature": TEMPERATURE,
    "alpha": ALPHA,
    "seed": 42,
    "total_parameters": total_student_params,
    "validation_loss": kd_val_loss,
    "validation_accuracy": kd_val_accuracy,
    "validation_f1": kd_val_f1
}

kd_results_path = os.path.join(
    RESULTS_DIR,
    "kd_results.json"
)

with open(
    kd_results_path,
    "w"
) as f:
    json.dump(
        kd_results,
        f,
        indent=4
    )

# ------------------------------------------
# Save KD training history
# ------------------------------------------

kd_history_path = os.path.join(
    RESULTS_DIR,
    "kd_training_history.json"
)

with open(
    kd_history_path,
    "w"
) as f:
    json.dump(
        kd_training_history,
        f,
        indent=4
    )

print("====================================")
print("KD RESULTS SAVED")
print("====================================")

print("KD results:")
print(kd_results_path)

print("\nKD training history:")
print(kd_history_path)

print("\nSaved KD result:")
print(json.dumps(
    kd_results,
    indent=4
))

print("\nFiles currently in results folder:")

for filename in sorted(
    os.listdir(RESULTS_DIR)
):
    print(filename)

KD RESULTS SAVED
KD results:
/content/drive/MyDrive/SLM_KD_Research/results/kd_results.json

KD training history:
/content/drive/MyDrive/SLM_KD_Research/results/kd_training_history.json

Saved KD result:
{
    "experiment": "E2_KD_student",
    "teacher_model": "textattack/bert-base-uncased-SST-2",
    "student_model": "google/bert_uncased_L-2_H-128_A-2",
    "dataset": "SST-2",
    "split": "validation",
    "validation_samples": 872,
    "max_length": 128,
    "batch_size": 16,
    "learning_rate": 2e-05,
    "epochs": 3,
    "temperature": 2.0,
    "alpha": 0.5,
    "seed": 42,
    "total_parameters": 4386178,
    "validation_loss": 0.5380656463856047,
    "validation_accuracy": 0.8188073394495413,
    "validation_f1": 0.8297413793103449
}

Files currently in results folder:
ablation
baseline_results.json
error_analysis
final
kd_results.json
kd_training_history.json
plots


In [ ]:
import os

target_name = "SLM_KD_Day2_Knowledge_Distillation.ipynb"

found = []

for root, dirs, files in os.walk("/content/drive/MyDrive"):
    if target_name in files:
        found.append(os.path.join(root, target_name))

if found:
    print("YES ✅ Notebook is saved in Google Drive.")
    for path in found:
        print("Location:", path)
else:
    print("NO ❌ Notebook was not found in Google Drive.")

YES ✅ Notebook is saved in Google Drive.
Location: /content/drive/MyDrive/Colab Notebooks/SLM_KD_Day2_Knowledge_Distillation.ipynb


In [ ]:
import os
import json

TARGET_NOTEBOOK = "SLM_KD_Day2_Knowledge_Distillation.ipynb"

found = []

for root, dirs, files in os.walk("/content/drive/MyDrive"):
    if TARGET_NOTEBOOK in files:
        found.append(os.path.join(root, TARGET_NOTEBOOK))

if not found:
    print("❌ NOTEBOOK NOT FOUND IN GOOGLE DRIVE")
else:
    print("✅ NOTEBOOK FOUND!")
    print()

    notebook_path = found[0]
    print("Location:")
    print(notebook_path)
    print()

    with open(notebook_path, "r", encoding="utf-8") as f:
        notebook = json.load(f)

    cells = notebook.get("cells", [])

    code_cells = [
        cell for cell in cells
        if cell.get("cell_type") == "code"
    ]

    print("Total cells:", len(cells))
    print("Total code cells:", len(code_cells))
    print()

    print("=" * 60)
    print("CODE CELLS IN THE SAVED DAY 2 NOTEBOOK")
    print("=" * 60)

    for i, cell in enumerate(code_cells, start=1):
        source = "".join(cell.get("source", []))

        print()
        print(f"========== CODE CELL {i} ==========")
        print(source[:2000])

        if len(source) > 2000:
            print("... [remaining code in this cell omitted from display]")

✅ NOTEBOOK FOUND!

Location:
/content/drive/MyDrive/Colab Notebooks/SLM_KD_Day2_Knowledge_Distillation.ipynb

Total cells: 28
Total code cells: 28

CODE CELLS IN THE SAVED DAY 2 NOTEBOOK

========== CODE CELL 1 ==========
# ==========================================
# DAY 2 — SYSTEM CHECK
# ==========================================

import torch
import os

print("====================================")
print("DAY 2 — SYSTEM CHECK")
print("====================================")

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
else:
    print("WARNING: GPU is not available.")

PROJECT_ROOT = "/content/drive/MyDrive/SLM_KD_Research"

print("\nProject root:")
print(PROJECT_ROOT)

print("\nBaseline checkpoint exists:",
      os.path.exists(
          os.path.join(
              PROJECT_ROOT,
              "models",
  

In [ ]:
import os

PROJECT_ROOT = "/content/drive/MyDrive/SLM_KD_Research"

print("Searching for KD-related saved files...\n")

for root, dirs, files in os.walk(PROJECT_ROOT):
    for file in files:
        if any(x in file.lower() for x in ["kd", "result", "history", "temperature", "alpha"]):
            print(os.path.join(root, file))

Searching for KD-related saved files...

/content/drive/MyDrive/SLM_KD_Research/results/baseline_results.json
/content/drive/MyDrive/SLM_KD_Research/results/kd_results.json
/content/drive/MyDrive/SLM_KD_Research/results/kd_training_history.json
/content/drive/MyDrive/SLM_KD_Research/results/master_results.csv
/content/drive/MyDrive/SLM_KD_Research/results/efficiency_results.csv
/content/drive/MyDrive/SLM_KD_Research/results/inference_results.csv
/content/drive/MyDrive/SLM_KD_Research/results/baseline_training_history.json
/content/drive/MyDrive/SLM_KD_Research/results/baseline_seed43_results.json
/content/drive/MyDrive/SLM_KD_Research/results/baseline_seed43_training_history.json
/content/drive/MyDrive/SLM_KD_Research/results/kd_seed43_results.json
/content/drive/MyDrive/SLM_KD_Research/results/kd_seed43_training_history.json
/content/drive/MyDrive/SLM_KD_Research/results/baseline_seed44_results.json
/content/drive/MyDrive/SLM_KD_Research/results/baseline_seed44_training_history.json
/

In [ ]:
import json
import os

files_to_check = [
    "/content/drive/MyDrive/SLM_KD_Research/results/kd_results.json",
    "/content/drive/MyDrive/SLM_KD_Research/results/kd_training_history.json",
]

for path in files_to_check:
    print("\n" + "="*70)
    print("FILE:", os.path.basename(path))
    print("="*70)

    with open(path, "r") as f:
        data = json.load(f)

    print(json.dumps(data, indent=2))


FILE: kd_results.json
{
  "experiment": "E2_KD_student",
  "teacher_model": "textattack/bert-base-uncased-SST-2",
  "student_model": "google/bert_uncased_L-2_H-128_A-2",
  "dataset": "SST-2",
  "split": "validation",
  "validation_samples": 872,
  "max_length": 128,
  "batch_size": 16,
  "learning_rate": 2e-05,
  "epochs": 3,
  "temperature": 2.0,
  "alpha": 0.5,
  "seed": 42,
  "total_parameters": 4386178,
  "validation_loss": 0.5380656463856047,
  "validation_accuracy": 0.8188073394495413,
  "validation_f1": 0.8297413793103449
}

FILE: kd_training_history.json
[
  {
    "epoch": 1,
    "total_loss": 0.8298704188609066,
    "ce_loss": 0.45688432181373484,
    "kd_loss": 1.2028565162115878,
    "time_seconds": 231.0123634338379
  },
  {
    "epoch": 2,
    "total_loss": 0.5259916934647073,
    "ce_loss": 0.3272341118902056,
    "kd_loss": 0.7247492746244275,
    "time_seconds": 223.54481148719788
  },
  {
    "epoch": 3,
    "total_loss": 0.42053299912488235,
    "ce_loss": 0.27285096

In [ ]:
# ==================================================
# STEP 23 - SAVE T=1.5 / ALPHA=0.4 KD CHECKPOINT
# ==================================================

import os

KD_CHECKPOINT_DIR = os.path.join(
    PROJECT_ROOT,
    "models",
    "kd",
    "T1.5_A0.4"
)

os.makedirs(
    KD_CHECKPOINT_DIR,
    exist_ok=True
)

kd_student_model.save_pretrained(
    KD_CHECKPOINT_DIR
)

student_tokenizer.save_pretrained(
    KD_CHECKPOINT_DIR
)

print("==================================================")
print("T=1.5 / ALPHA=0.4 KD CHECKPOINT SAVED")
print("==================================================")

print("Checkpoint path:")
print(KD_CHECKPOINT_DIR)

print("\nFiles saved:")

for filename in sorted(
    os.listdir(KD_CHECKPOINT_DIR)
):
    print(filename)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

T=1.5 / ALPHA=0.4 KD CHECKPOINT SAVED
Checkpoint path:
/content/drive/MyDrive/SLM_KD_Research/models/kd/T1.5_A0.4

Files saved:
config.json
model.safetensors
tokenizer.json
tokenizer_config.json


In [ ]:
# ==================================================
# STEP 24 - SAVE T=1.5 / ALPHA=0.4 RESULTS
# ==================================================

import os
import json

RESULTS_DIR = os.path.join(
    PROJECT_ROOT,
    "results"
)

os.makedirs(
    RESULTS_DIR,
    exist_ok=True
)

kd_t15_results = {
    "experiment": "E2_KD_student",
    "teacher_model": "textattack/bert-base-uncased-SST-2",
    "student_model": "google/bert_uncased_L-2_H-128_A-2",
    "dataset": "SST-2",
    "split": "validation",
    "validation_samples": 872,
    "max_length": 128,
    "batch_size": 16,
    "learning_rate": "2e-05",
    "epochs": 3,
    "temperature": 1.5,
    "alpha": 0.4,
    "seed": 42,
    "total_parameters": 4386178,
    "validation_loss": 0.6292748449878259,
    "validation_accuracy": 0.8199541284449367,
    "validation_f1": 0.8302702702702702
}

RESULT_PATH = os.path.join(
    RESULTS_DIR,
    "kd_T1.5_A0.4_results.json"
)

with open(
    RESULT_PATH,
    "w"
) as f:
    json.dump(
        kd_t15_results,
        f,
        indent=4
    )

print("==================================================")
print("T=1.5 / ALPHA=0.4 RESULTS SAVED")
print("==================================================")

print("Result file:")
print(RESULT_PATH)

print("\nSaved values:")
print("Accuracy:", kd_t15_results["validation_accuracy"])
print("F1:", kd_t15_results["validation_f1"])
print("Temperature:", kd_t15_results["temperature"])
print("Alpha:", kd_t15_results["alpha"])

T=1.5 / ALPHA=0.4 RESULTS SAVED
Result file:
/content/drive/MyDrive/SLM_KD_Research/results/kd_T1.5_A0.4_results.json

Saved values:
Accuracy: 0.8199541284449366
F1: 0.8302702702702702
Temperature: 1.5
Alpha: 0.4


In [ ]:
# ==================================================
# STEP 25 - SAVE T=1.5 / ALPHA=0.4 TRAINING HISTORY
# ==================================================

import os
import json

HISTORY_PATH = os.path.join(
    PROJECT_ROOT,
    "results",
    "kd_T1.5_A0.4_training_history.json"
)

with open(HISTORY_PATH, "w") as f:
    json.dump(
        kd_training_history,
        f,
        indent=4
    )

print("==================================================")
print("T=1.5 / ALPHA=0.4 TRAINING HISTORY SAVED")
print("==================================================")

print("History file:")
print(HISTORY_PATH)

print("\nNumber of epochs:")
print(len(kd_training_history))

print("\nTraining history:")
for epoch in kd_training_history:
    print(epoch)

T=1.5 / ALPHA=0.4 TRAINING HISTORY SAVED
History file:
/content/drive/MyDrive/SLM_KD_Research/results/kd_T1.5_A0.4_training_history.json

Number of epochs:
3

Training history:
{'epoch': 1, 'total_loss': 0.2681914729608174, 'ce_loss': 0.2238379497934938, 'kd_loss': 0.33472173756913165, 'time_seconds': 197.87132811546326}
{'epoch': 2, 'total_loss': 0.23444957410332673, 'ce_loss': 0.1995878492143545, 'kd_loss': 0.28674214436670414, 'time_seconds': 203.89971470832825}
{'epoch': 3, 'total_loss': 0.21002380025137277, 'ce_loss': 0.181365246253504, 'kd_loss': 0.25301161588494825, 'time_seconds': 205.15709161758423}


In [ ]:
# ==================================================
# SAVE T=1.0 / ALPHA=0.3 TRAINING HISTORY
# ==================================================

import os
import json

T1_HISTORY = [
    {
        "epoch": 1,
        "total_loss": 0.4317066929190289,
        "ce_loss": 0.443884819269959,
        "kd_loss": 0.4032910644441772,
        "time_seconds": 200.8543689250946
    },
    {
        "epoch": 2,
        "total_loss": 0.2958112240138501,
        "ce_loss": 0.30869756516272695,
        "kd_loss": 0.2657430953551924,
        "time_seconds": 204.8556501865387
    },
    {
        "epoch": 3,
        "total_loss": 0.24155122979882565,
        "ce_loss": 0.2545318366486287,
        "kd_loss": 0.21126314818752917,
        "time_seconds": 205.37742853164673
    }
]

PATH = os.path.join(
    PROJECT_ROOT,
    "results",
    "kd_T1.0_A0.3_training_history.json"
)

with open(PATH, "w") as f:
    json.dump(T1_HISTORY, f, indent=4)

print("==================================================")
print("T=1.0 / ALPHA=0.3 TRAINING HISTORY SAVED")
print("==================================================")
print(PATH)

T=1.0 / ALPHA=0.3 TRAINING HISTORY SAVED
/content/drive/MyDrive/SLM_KD_Research/results/kd_T1.0_A0.3_training_history.json


In [ ]:
# ==================================================
# SAVE T=2.0 / ALPHA=0.5 TRAINING HISTORY
# ==================================================

import os
import json

T2_HISTORY = [
    {
        "epoch": 1,
        "total_loss": 0.8298704188609066,
        "ce_loss": 0.45688432181373484,
        "kd_loss": 1.2028565162115878,
        "time_seconds": 231.0123634338379
    },
    {
        "epoch": 2,
        "total_loss": 0.5259916934647073,
        "ce_loss": 0.3272341118902056,
        "kd_loss": 0.7247492746244275,
        "time_seconds": 223.54481148719788
    },
    {
        "epoch": 3,
        "total_loss": 0.42053299912488235,
        "ce_loss": 0.2728509630307759,
        "kd_loss": 0.5682150349251024,
        "time_seconds": 224.16332578659058
    }
]

PATH = os.path.join(
    PROJECT_ROOT,
    "results",
    "kd_T2.0_A0.5_training_history.json"
)

with open(PATH, "w") as f:
    json.dump(T2_HISTORY, f, indent=4)

print("==================================================")
print("T=2.0 / ALPHA=0.5 TRAINING HISTORY SAVED")
print("==================================================")
print(PATH)

T=2.0 / ALPHA=0.5 TRAINING HISTORY SAVED
/content/drive/MyDrive/SLM_KD_Research/results/kd_T2.0_A0.5_training_history.json


In [ ]:
T15_HISTORY = [
    {
        "epoch": 1,
        "total_loss": 0.2681914729608174,
        "ce_loss": 0.2238379497934938,
        "kd_loss": 0.3347213756913165,
        "time_seconds": 197.87132811546326
    },
    {
        "epoch": 2,
        "total_loss": 0.23444957410332673,
        "ce_loss": 0.1995878492143545,
        "kd_loss": 0.28674214436670414,
        "time_seconds": 203.89971470832825
    },
    {
        "epoch": 3,
        "total_loss": 0.21002380025137277,
        "ce_loss": 0.181365246253504,
        "kd_loss": 0.25301161588494825,
        "time_seconds": 205.15709161758423
    }
]

PATH = os.path.join(
    PROJECT_ROOT,
    "results",
    "kd_T1.5_A0.4_training_history.json"
)

with open(PATH, "w") as f:
    json.dump(T15_HISTORY, f, indent=4)

print("==================================================")
print("T=1.5 / ALPHA=0.4 TRAINING HISTORY SAVED")
print("==================================================")
print(PATH)

T=1.5 / ALPHA=0.4 TRAINING HISTORY SAVED
/content/drive/MyDrive/SLM_KD_Research/results/kd_T1.5_A0.4_training_history.json


In [ ]:
T15_HISTORY = [
    {
        "epoch": 1,
        "total_loss": 0.2681914729608174,
        "ce_loss": 0.2238379497934938,
        "kd_loss": 0.3347213756913165,
        "time_seconds": 197.87132811546326
    },
    {
        "epoch": 2,
        "total_loss": 0.23444957410332673,
        "ce_loss": 0.1995878492143545,
        "kd_loss": 0.28674214436670414,
        "time_seconds": 203.89971470832825
    },
    {
        "epoch": 3,
        "total_loss": 0.21002380025137277,
        "ce_loss": 0.181365246253504,
        "kd_loss": 0.25301161588494825,
        "time_seconds": 205.15709161758423
    }
]

PATH = os.path.join(
    PROJECT_ROOT,
    "results",
    "kd_T1.5_A0.4_training_history.json"
)

with open(PATH, "w") as f:
    json.dump(T15_HISTORY, f, indent=4)

print("==================================================")
print("T=1.5 / ALPHA=0.4 TRAINING HISTORY SAVED")
print("==================================================")
print(PATH)

T=1.5 / ALPHA=0.4 TRAINING HISTORY SAVED
/content/drive/MyDrive/SLM_KD_Research/results/kd_T1.5_A0.4_training_history.json


In [ ]:
# ==================================================
# SAVE T=2.0 / ALPHA=0.5 RESULTS
# ==================================================

import os
import json

T2_RESULTS = {
    "experiment": "E2_KD_student",
    "teacher_model": "textattack/bert-base-uncased-SST-2",
    "student_model": "google/bert_uncased_L-2_H-128_A-2",
    "dataset": "SST-2",
    "split": "validation",
    "validation_samples": 872,
    "max_length": 128,
    "batch_size": 16,
    "learning_rate": "2e-05",
    "epochs": 3,
    "temperature": 2.0,
    "alpha": 0.5,
    "seed": 42,
    "total_parameters": 4386178,
    "validation_loss": 0.5380656463856047,
    "validation_accuracy": 0.8188073394495413,
    "validation_f1": 0.8297413793103449
}

PATH = os.path.join(
    PROJECT_ROOT,
    "results",
    "kd_T2.0_A0.5_results.json"
)

with open(PATH, "w") as f:
    json.dump(T2_RESULTS, f, indent=4)

print("==================================================")
print("T=2.0 / ALPHA=0.5 RESULTS SAVED")
print("==================================================")
print(PATH)

T=2.0 / ALPHA=0.5 RESULTS SAVED
/content/drive/MyDrive/SLM_KD_Research/results/kd_T2.0_A0.5_results.json
